# Modelamiento de Datos

Este notebook implementa un análisis completo de clasificación de riesgo utilizando técnicas de machine learning con interpretabilidad basada en SHAP.

In [ ]:
#Base de datos preprocesada

### Análisis de Desbalance en la Variable Objetivo
**IMPORTANTE**: Este análisis identifica si existe desbalance severo en la variable objetivo `M3_30AC` (Riesgo de Mora).
Un desbalance severo afectará el rendimiento del modelo y requerirá técnicas de balanceo. 

In [ ]:
# ──  Distribución de la variable objetivo ───────────────────────────────
print("DISTRIBUCIÓN VARIABLE OBJETIVO (M3_30AC)")
conteo = df_financiacion['M3_30AC'].value_counts()
pct_target = (conteo / len(df_financiacion) * 100).round(2)
print(pd.DataFrame({'Conteo': conteo, 'Porcentaje (%)': pct_target}))

## PASO 4: Función de Muestreo Estratificado y Codificación de Variables

### Descripción
Se define una función que genera **muestras aleatorias diferentes** del dataset original (sin balancear), manteniendo la proporción de clases. El balanceo se aplicará a cada muestra en la Sección 2.

**Ventajas de este enfoque:**
- Cada muestra selecciona clientes DIFERENTES de la clase minoritaria
- Valida robustez del modelo con múltiples corridas
- Mayor variabilidad y representatividad
- Cambiar `random_state` genera nuevas muestras diferentes

In [ ]:
def generar_muestra_estratificada(df, proporcion=1.0, random_state=42):
    """
    Genera una muestra aleatoria estratificada preservando la proporción de clases.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame original (con o sin balance)
    proporcion : float (default=1.0)
        Proporción de datos a muestrear (0.0 a 1.0)
        - 0.8 = 80% de los datos
        - 1.0 = 100% de los datos
    random_state : int (default=42)
        Semilla aleatoria para reproducibilidad
        Cambiar este valor genera muestras diferentes
    
    Retorna:
    --------
    pd.DataFrame : Muestra estratificada del dataset
    
    Nota:
    -----
    Esta función se usa con df_financiacion (original).
    El balanceo se aplica DESPUÉS a cada muestra en la Sección 2.
    """
    from sklearn.model_selection import train_test_split
    
    if not (0 < proporcion <= 1.0):
        raise ValueError("La proporción debe estar entre 0 y 1")
    
    if proporcion == 1.0:
        return df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    
    muestra, _ = train_test_split(
        df,
        test_size=(1 - proporcion),
        stratify=df['M3_30AC'],
        random_state=random_state
    )
    
    return muestra.reset_index(drop=True)


# ── Demostración de la función ────────────────────────────────────────────
print("FUNCIÓN DE MUESTREO ESTRATIFICADO")
print("=" * 70)
print(f"\nDataset original (SIN balancear):")
print(f"   Total: {len(df_financiacion)} registros")
print(f"   Sin mora (0): {(df_financiacion['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(df_financiacion['M3_30AC'] == 1).sum()}")
print(f"   Proporción: {(df_financiacion['M3_30AC'] == 0).sum() / (df_financiacion['M3_30AC'] == 1).sum():.1f}:1 (DESBALANCEADO)")

muestra_demo = generar_muestra_estratificada(df_financiacion, proporcion=1.0, random_state=42)
print(f"\nMuestra estratificada (100%, random_state=42):")
print(f"   Total: {len(muestra_demo)} registros")
print(f"   Sin mora (0): {(muestra_demo['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(muestra_demo['M3_30AC'] == 1).sum()}")
print(f"   Proporción: {(muestra_demo['M3_30AC'] == 0).sum() / (muestra_demo['M3_30AC'] == 1).sum():.1f}:1 (MANTIENE DESBALANCE)")

print("\nFunción lista para usar en la Sección 2")
print("=" * 70)

## PASO 5: Codificación de Variables Categóricas

### Descripción
Se convierten las variables categóricas binarias en formato numérico. Esta codificación se aplicará al dataset original antes de cualquier muestreo.

**Variables a codificar:**
- `Genero`: MASCULINO = 1, FEMENINO = 0
- `Perfil`: ASALARIADO = 1, INDEPENDIENTE = 0  
- `Estado`: NUEVO = 1, USADO = 0

In [ ]:
# Verificar los valores únicos de variables categóricas antes de codificar
print("VALORES ÚNICOS EN VARIABLES CATEGÓRICAS")
print("=" * 50)
print(f"Genero: {df_financiacion['Genero'].unique()}")
print(f"Perfil: {df_financiacion['Perfil'].unique()}")
print(f"Estado: {df_financiacion['Estado'].unique()}")
print("=" * 50)

# ── Codificación de variables categóricas (binarias) ──────────────────────
df_financiacion['Genero']  = df_financiacion['Genero'].map({'MASCULINO': 1, 'FEMENINO': 0})
df_financiacion['Perfil']  = df_financiacion['Perfil'].map({'ASALARIADO': 1, 'INDEPENDIENTE': 0})
df_financiacion['Estado']  = df_financiacion['Estado'].map({'NUEVO': 1, 'USADO': 0})

print(" Codificación aplicada correctamente")
print(f"Verificación - Valores únicos después de codificar:")
print(f"Genero: {df_financiacion['Genero'].unique()}")
print(f"Perfil: {df_financiacion['Perfil'].unique()}")
print(f"Estado: {df_financiacion['Estado'].unique()}")

# Visualizar primeras filas del dataset codificado
print("Primeras filas del dataset después de codificar variables:")
df_financiacion.head()

# ── Eliminar variables que no aportan información ─────────────────────────
# Se elimina la columna 'Caso' que es solo un identificador de fila
df_financiacion = df_financiacion.drop(columns=['Caso'])

print("  Dataset preparado para muestreo y modelamiento")
print(f"\n Resumen final:")
print(f"   Total de registros: {len(df_financiacion)}")
print(f"   Total de variables: {df_financiacion.shape[1]}")
print(f"   Sin mora (Clase 0): {(df_financiacion['M3_30AC'] == 0).sum()}")
print(f"   Con mora (Clase 1): {(df_financiacion['M3_30AC'] == 1).sum()}")
print(f"   Proporción original: {(df_financiacion['M3_30AC'] == 0).sum() / (df_financiacion['M3_30AC'] == 1).sum():.1f}:1 (DESBALANCEADO)")
print(f"\n    El balanceo se aplicará a cada muestra en la Sección 2")

---

## Resumen de la Preparación de Datos

### Transformaciones Realizadas:
- **Datos nulos**: Eliminadas filas con valores faltantes (0.13%)
- **Datos duplicados**: Verificación completada
- **Codificación**: Convertidas variables categóricas a numéricas
- **Limpieza**: Eliminada columna ID innecesaria
- **Función de muestreo**: Definida para generar muestras estratificadas

### Dataset Preparado (SIN BALANCEAR AÚN):
- **Total de registros**: 21,330 observaciones
- **Total de variables**: 10 features + 1 target
- **Variable objetivo**: M3_30AC (Sin mora=0, Con mora=1)
- **Desbalance**: Proporción 24:1 (20,493 sin mora, 837 con mora)

### Estrategia para la Sección 2:
1. **Generar múltiples muestras** con `generar_muestra_estratificada()`
   - Cambiar `random_state` para cada corrida
   - Mantiene proporción de clases en cada muestra

2. **Aplicar balanceo a cada muestra** con el ratio que elijas
   - Opciones sugeridas: `1:1`, `1:2` o `1:3`
   - El valor se ajusta en `aplicar_balanceo(..., ratio=...)`

3. **Dividir la muestra balanceada** en entrenamiento y prueba
   - Esquema sugerido para pocos datos: `80/20`
   - El valor se ajusta en `preparar_corrida_experimento(..., proporciones_split=...)`

4. **Entrenar modelos** con cada muestra balanceada
   - Regresión Logística (con escalamiento)
   - XGBoost (sin escalamiento)
   - Validar robustez con múltiples corridas

---

# SECCIÓN 2: MODELAMIENTO Y CLASIFICACIÓN

En esta sección se desarrollarán, optimizarán e interpretarán modelos de machine learning para predecir el riesgo de mora en clientes.

**Flujo de trabajo:**
1. Generar muestra estratificada con `random_state`
2. Aplicar balanceo con el ratio elegido
3. Dividir en train/test
4. Entrenar Regresión Logística y XGBoost
5. Comparar desempeño de modelos
6. Análisis SHAP para interpretabilidad
7. Validación manual con 10 clientes ficticios

## Importar Librerías para Modelamiento

### Descripción
Se importan las librerías necesarias de scikit-learn, XGBoost y utilidades para modelamiento.

In [ ]:
# ── Librerías para modelamiento ──────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report, roc_curve, auc)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
warnings.filterwarnings('ignore')

## Función para Aplicar Balanceo a Cada Muestra

### Descripción
Se define una función que aplica balanceo mediante undersampling 1:2 a una muestra dada. Esta función se usará en cada corrida con diferentes muestras.

In [1]:
def aplicar_balanceo(df, ratio=2, random_state=42):
    """
    Aplica balanceo mediante undersampling con ratio clase minoritaria:clase mayoritaria.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a balancear
    ratio : int (default=2)
        Ratio de balanceo: clase_mayoritaria / clase_minoritaria
        - ratio=2 → 1:2 (1 caso con mora, 2 sin mora)
    random_state : int (default=42)
        Semilla aleatoria para reproducibilidad
    
    Retorna:
    --------
    pd.DataFrame : DataFrame balanceado
    """
    
    clase_1 = df[df['M3_30AC'] == 1]
    clase_0 = df[df['M3_30AC'] == 0]
    
    # Muestrear clase mayoritaria (0) según ratio
    n_clase_0 = len(clase_1) * ratio
    clase_0_muestra = clase_0.sample(n=min(n_clase_0, len(clase_0)), random_state=random_state)
    
    # Combinar y mezclar
    df_balanceado = pd.concat([clase_1, clase_0_muestra]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return df_balanceado


# ── Demostración de la función ────────────────────────────────────────────
print("FUNCIÓN DE BALANCEO (Undersampling 1:2)")
print("=" * 70)

# Generar una muestra de ejemplo
muestra_demo = generar_muestra_estratificada(df_financiacion, random_state=42)
print(f"\n Antes de balanceo:")
print(f"   Total: {len(muestra_demo)}")
print(f"   Sin mora (0): {(muestra_demo['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(muestra_demo['M3_30AC'] == 1).sum()}")
print(f"   Proporción: {(muestra_demo['M3_30AC'] == 0).sum() / (muestra_demo['M3_30AC'] == 1).sum():.1f}:1")

# Aplicar balanceo
muestra_balanceada = aplicar_balanceo(muestra_demo, ratio=2, random_state=42)
print(f"\n Después de balanceo (ratio 1:2):")
print(f"   Total: {len(muestra_balanceada)}")
print(f"   Sin mora (0): {(muestra_balanceada['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(muestra_balanceada['M3_30AC'] == 1).sum()}")
print(f"   Proporción: {(muestra_balanceada['M3_30AC'] == 0).sum() / (muestra_balanceada['M3_30AC'] == 1).sum():.1f}:1")

print("\n Función lista para usar en cada corrida")
print("=" * 70)

FUNCIÓN DE BALANCEO (Undersampling 1:2)


NameError: name 'generar_muestra_estratificada' is not defined

In [ ]:
def preparar_corrida_experimento(df, proporcion_muestra=1.0, ratio_balanceo=2, proporciones_split=(0.8, 0.2), random_state=42):
    """
    Prepara una corrida completa: muestra estratificada, balanceo y partición train/validación/test.

    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame original con la variable objetivo M3_30AC.
    proporcion_muestra : float
        Proporción de la base original a muestrear.
    ratio_balanceo : int
        Ratio clase mayoritaria / clase minoritaria. Ej.: 1, 2 o 3.
    proporciones_split : tuple[float, float] | tuple[float, float, float]
        Proporciones de train/test o train/validación/test. Deben sumar 1.0.
    random_state : int
        Semilla aleatoria para reproducibilidad.

    Retorna:
    --------
    dict
        Contiene la muestra, el balanceo y los conjuntos train/test o train/validación/test.
    """
    import numpy as np
    from sklearn.model_selection import train_test_split

    if len(proporciones_split) not in (2, 3):
        raise ValueError("proporciones_split debe tener 2 valores (train, test) o 3 valores (train, validación, test)")

    if not np.isclose(sum(proporciones_split), 1.0):
        raise ValueError("La suma de las proporciones debe ser 1.0")

    if any(proporcion <= 0 for proporcion in proporciones_split):
        raise ValueError("Todas las proporciones de split deben ser mayores que 0")

    df_muestra = generar_muestra_estratificada(
        df,
        proporcion=proporcion_muestra,
        random_state=random_state
    )

    df_balanceado = aplicar_balanceo(
        df_muestra,
        ratio=ratio_balanceo,
        random_state=random_state
    )

    X = df_balanceado.drop(columns=['M3_30AC'])
    y = df_balanceado['M3_30AC']

    if len(proporciones_split) == 2:
        train_prop, test_prop = proporciones_split
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=test_prop,
            stratify=y,
            random_state=random_state
        )

        return {
            'df_muestra': df_muestra,
            'df_balanceado': df_balanceado,
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test,
        }

    train_prop, val_prop, test_prop = proporciones_split
    val_test_prop = val_prop + test_prop

    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X,
        y,
        test_size=val_test_prop,
        stratify=y,
        random_state=random_state
    )

    test_rel = test_prop / val_test_prop
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp,
        y_tmp,
        test_size=test_rel,
        stratify=y_tmp,
        random_state=random_state
    )

    return {
        'df_muestra': df_muestra,
        'df_balanceado': df_balanceado,
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
    }


## CORRIDA 1: Configuración y División Train/Test

### Descripción
Se configura la primera corrida con `random_state=42`, se genera la muestra, se aplica balanceo y se divide en conjuntos de entrenamiento y prueba.

**Parámetros de la corrida:**
- `random_state=42`: Semilla para reproducibilidad
- `proporciones_split=(0.8, 0.2)`: 80% entrenamiento, 20% prueba
- `stratify`: División estratificada por clase

In [ ]:
# ── CORRIDA 1: Generar muestra, balancear y dividir ──────────────────────
RANDOM_STATE_CORRIDA = 42
PROPORCION_MUESTRA = 1.0
RATIO_BALANCEO = 2
PROPORCIONES_SPLIT = (0.8, 0.2)  # Puede probar (0.7, 0.3)

corrida = preparar_corrida_experimento(
    df_financiacion,
    proporcion_muestra=PROPORCION_MUESTRA,
    ratio_balanceo=RATIO_BALANCEO,
    proporciones_split=PROPORCIONES_SPLIT,
    random_state=RANDOM_STATE_CORRIDA
)

# Variables principales de trabajo

df_muestra = corrida['df_muestra']
df_datos = corrida['df_balanceado']
X_train = corrida['X_train']
X_test = corrida['X_test']
y_train = corrida['y_train']
y_test = corrida['y_test']
X_val = corrida.get('X_val')
y_val = corrida.get('y_val')

print("CONFIGURACIÓN DE LA CORRIDA 1")
print("=" * 70)
print(f"\n Parámetros usados:")
print(f"   random_state: {RANDOM_STATE_CORRIDA}")
print(f"   proporción de muestra: {PROPORCION_MUESTRA}")
print(f"   ratio de balanceo: 1:{RATIO_BALANCEO}")
print(f"   split train/test: {PROPORCIONES_SPLIT[0]:.0%}/{PROPORCIONES_SPLIT[1]:.0%}")

print(f"\n Dataset balanceado:")
print(f"   Total de registros: {len(df_datos)}")
print(f"   Sin mora (0): {(df_datos['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(df_datos['M3_30AC'] == 1).sum()}")
print(f"   Proporción: 1:{(df_datos['M3_30AC'] == 0).sum() / (df_datos['M3_30AC'] == 1).sum():.2f}")

print(f"\n División Train/Test:")
print(f"   Entrenamiento: {len(X_train)} registros ({len(X_train)/len(df_datos)*100:.1f}%)")
print(f"      - Sin mora: {(y_train == 0).sum()}")
print(f"      - Con mora: {(y_train == 1).sum()}")
print(f"   Prueba: {len(X_test)} registros ({len(X_test)/len(df_datos)*100:.1f}%)")
print(f"      - Sin mora: {(y_test == 0).sum()}")
print(f"      - Con mora: {(y_test == 1).sum()}")

print("\n Datos preparados para entrenamiento y prueba")
print("=" * 70)


## MODELO 1: Regresión Logística (CON Escalamiento)

### Descripción
Se entrena un modelo de Regresión Logística con escalamiento StandardScaler. La regresión logística es un modelo lineal que asume relaciones lineales entre features y probabilidad de riesgo.

In [ ]:
# Entrenar Regresión Logística (BASE) + métricas en TRAIN y TEST
print("MODELO 1: REGRESIÓN LOGÍSTICA (BASE)")
print("=" * 70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE_CORRIDA)
lr_model.fit(X_train_scaled, y_train)

# Predicciones TRAIN
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_proba_lr_train = lr_model.predict_proba(X_train_scaled)[:, 1]

# Predicciones TEST
y_pred_lr = lr_model.predict(X_test_scaled)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Métricas TRAIN y TEST
resumen_lr = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precisión', 'Recall', 'F1-Score', 'ROC-AUC'],
    'TRAIN': [
        accuracy_score(y_train, y_pred_lr_train),
        precision_score(y_train, y_pred_lr_train),
        recall_score(y_train, y_pred_lr_train),
        f1_score(y_train, y_pred_lr_train),
        roc_auc_score(y_train, y_proba_lr_train)
    ],
    'TEST': [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        roc_auc_score(y_test, y_proba_lr)
    ]
})

print(resumen_lr.to_string(index=False))
print("=" * 70)

# Matriz de confusión TEST
cm_test = confusion_matrix(y_test, y_pred_lr, labels=lr_model.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=lr_model.classes_)
disp_test.plot(cmap='Oranges', ax=ax, colorbar=False)
ax.set_title('Confusion Matrix - TEST')
plt.tight_layout()
plt.show()

---

## MODELO  2: XGBoost (Base) 

### Descripción
Se entrena un modelo XGBoost (eXtreme Gradient Boosting) 
XGBoost es un modelo basado en árboles que puede capturar relaciones no lineales y es muy potente.

In [ ]:
print("\nMODELO 2: XGBOOST (BASE)")
print("=" * 70)

xgb_model = XGBClassifier(random_state=RANDOM_STATE_CORRIDA, eval_metric='logloss', verbose=0)
xgb_model.fit(X_train, y_train)

# Predicciones TRAIN
y_pred_xgb_train = xgb_model.predict(X_train)
y_proba_xgb_train = xgb_model.predict_proba(X_train)[:, 1]

# Predicciones TEST
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
# Métricas TRAIN y TEST
resumen_xgb = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precisión', 'Recall', 'F1-Score', 'ROC-AUC'],
    'TRAIN': [
        accuracy_score(y_train, y_pred_xgb_train),
        precision_score(y_train, y_pred_xgb_train),
        recall_score(y_train, y_pred_xgb_train),
        f1_score(y_train, y_pred_xgb_train),
        roc_auc_score(y_train, y_proba_xgb_train)
    ],
    'TEST': [
        accuracy_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb),
        roc_auc_score(y_test, y_proba_xgb)
    ]
})

print(resumen_xgb.to_string(index=False))
print("=" * 70)

In [ ]:
def construir_comparativa_modelos(y_train, y_test, modelos, orden_modelos=None):
    """
    Construye una tabla comparativa de métricas TRAIN y TEST para varios modelos.

    La tabla se organiza por bloques de modelo, con columnas:
    Métrica, Valor TRAIN, Valor TEST.

    Parámetros:
    -----------
    y_train, y_test : array-like
        Etiquetas reales de entrenamiento y prueba.
    modelos : dict
        Diccionario con la estructura:
        {
            'Nombre modelo': {
                'y_pred_train': ..., 'y_proba_train': ...,
                'y_pred_test': ...,  'y_proba_test': ...
            }
        }
    orden_modelos : list[str] | None
        Orden en que se imprimirán los modelos.
        Si no se pasa, se usa el orden del diccionario.

    Retorna:
    --------
    pd.DataFrame
        Tabla con columnas: Modelo, Métrica, Valor TRAIN, Valor TEST.
    """
    import numpy as np

    def safe_metric(func, *args):
        try:
            return func(*args)
        except Exception:
            return np.nan

    metricas = ['Accuracy', 'Precisión', 'Recall', 'F1-Score', 'ROC-AUC']
    filas = []

    if orden_modelos is None:
        orden_modelos = list(modelos.keys())

    for nombre_modelo in orden_modelos:
        datos = modelos.get(nombre_modelo)
        if datos is None:
            continue

        valores_train = [
            safe_metric(accuracy_score, y_train, datos['y_pred_train']),
            safe_metric(precision_score, y_train, datos['y_pred_train']),
            safe_metric(recall_score, y_train, datos['y_pred_train']),
            safe_metric(f1_score, y_train, datos['y_pred_train']),
            safe_metric(roc_auc_score, y_train, datos['y_proba_train']),
        ]
        valores_test = [
            safe_metric(accuracy_score, y_test, datos['y_pred_test']),
            safe_metric(precision_score, y_test, datos['y_pred_test']),
            safe_metric(recall_score, y_test, datos['y_pred_test']),
            safe_metric(f1_score, y_test, datos['y_pred_test']),
            safe_metric(roc_auc_score, y_test, datos['y_proba_test']),
        ]

        for metrica, valor_train, valor_test in zip(metricas, valores_train, valores_test):
            filas.append({
                'Modelo': nombre_modelo,
                'Métrica': metrica,
                'Valor TRAIN': valor_train,
                'Valor TEST': valor_test,
            })

    return pd.DataFrame(filas)

## Comparación de Modelos

Tabla comparativa de los dos modelos en su forma base y optimizada.

In [ ]:
print("\nCOMPARACIÓN - MODELOS BASE")
print("=" * 80)

modelos_base = {
    'Reg. Logística': {
        'y_pred_train': y_pred_lr_train,
        'y_proba_train': y_proba_lr_train,
        'y_pred_test': y_pred_lr,
        'y_proba_test': y_proba_lr,
    },
    'XGBoost': {
        'y_pred_train': y_pred_xgb_train,
        'y_proba_train': y_proba_xgb_train,
        'y_pred_test': y_pred_xgb,
        'y_proba_test': y_proba_xgb,
    },
}

comparativa_base = construir_comparativa_modelos(
    y_train=y_train,
    y_test=y_test,
    modelos=modelos_base,
    orden_modelos=['Reg. Logística', 'XGBoost']
)

for nombre_modelo in ['Reg. Logística', 'XGBoost']:
    bloque = comparativa_base[comparativa_base['Modelo'] == nombre_modelo]
    print(f"\n{nombre_modelo}")
    print("-" * 80)
    print(bloque[['Métrica', 'Valor TRAIN', 'Valor TEST']].to_string(index=False))

print("=" * 80)

---

## Optimización de Hiperparámetros con RandomizedSearchCV

Se optimizan ambos modelos buscando los mejores hiperparámetros usando RandomizedSearchCV con validación cruzada.


In [ ]:
# RandomizedSearchCV - Regresión Logística
print("\nOPTIMIZACIÓN: REGRESIÓN LOGÍSTICA")
print("=" * 70)

param_dist_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [500, 1000, 1500]
}

random_search_lr = RandomizedSearchCV(
    LogisticRegression(),
    param_dist_lr,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    random_state=RANDOM_STATE_CORRIDA,
    n_jobs=-1,
    verbose=0
)

random_search_lr.fit(X_train_scaled, y_train)

print(f"Mejores parámetros: {random_search_lr.best_params_}")
print(f"ROC-AUC (CV): {random_search_lr.best_score_:.4f}")

# Evaluar en prueba
lr_best = random_search_lr.best_estimator_
y_pred_lr_opt = lr_best.predict(X_test_scaled)
y_proba_lr_opt = lr_best.predict_proba(X_test_scaled)[:, 1]

# Evaluar en entrenamiento para la comparación TRAIN/TEST
y_pred_lr_opt_train = lr_best.predict(X_train_scaled)
y_proba_lr_opt_train = lr_best.predict_proba(X_train_scaled)[:, 1]

acc_lr_opt = accuracy_score(y_test, y_pred_lr_opt)
prec_lr_opt = precision_score(y_test, y_pred_lr_opt)
rec_lr_opt = recall_score(y_test, y_pred_lr_opt)
f1_lr_opt = f1_score(y_test, y_pred_lr_opt)
roc_lr_opt = roc_auc_score(y_test, y_proba_lr_opt)

In [ ]:
# Métricas TRAIN y TEST
resumen_lr = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precisión', 'Recall', 'F1-Score', 'ROC-AUC'],
    'TRAIN': [
        accuracy_score(y_train, y_pred_lr_opt_train),
        precision_score(y_train, y_pred_lr_opt_train),
        recall_score(y_train, y_pred_lr_opt_train),
        f1_score(y_train, y_pred_lr_opt_train),
        roc_auc_score(y_train, y_proba_lr_opt_train)
    ],
    'TEST': [
        accuracy_score(y_test, y_pred_lr_opt),
        precision_score(y_test, y_pred_lr_opt),
        recall_score(y_test, y_pred_lr_opt),
        f1_score(y_test, y_pred_lr_opt),
        roc_auc_score(y_test, y_proba_lr_opt)
    ]
})

print(resumen_lr.to_string(index=False))
print("=" * 70)

In [ ]:
# RandomizedSearchCV - XGBoost
print("\nOPTIMIZACIÓN: XGBOOST")
print("=" * 70)

param_dist_xgb = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9]
}

random_search_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=RANDOM_STATE_CORRIDA, eval_metric='logloss', verbose=0),
    param_dist_xgb,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    random_state=RANDOM_STATE_CORRIDA,
    n_jobs=-1,
    verbose=0
)

random_search_xgb.fit(X_train, y_train)

print(f"Mejores parámetros: {random_search_xgb.best_params_}")
print(f"ROC-AUC (CV): {random_search_xgb.best_score_:.4f}")

# Evaluar en prueba
xgb_best = random_search_xgb.best_estimator_
y_pred_xgb_opt = xgb_best.predict(X_test)
y_proba_xgb_opt = xgb_best.predict_proba(X_test)[:, 1]

# Evaluar en entrenamiento para la comparación TRAIN/TEST
y_pred_xgb_opt_train = xgb_best.predict(X_train)
y_proba_xgb_opt_train = xgb_best.predict_proba(X_train)[:, 1]

acc_xgb_opt = accuracy_score(y_test, y_pred_xgb_opt)
prec_xgb_opt = precision_score(y_test, y_pred_xgb_opt)
rec_xgb_opt = recall_score(y_test, y_pred_xgb_opt)
f1_xgb_opt = f1_score(y_test, y_pred_xgb_opt)
roc_xgb_opt = roc_auc_score(y_test, y_proba_xgb_opt)


In [ ]:
# Métricas TRAIN y TEST
resumen_xgb = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precisión', 'Recall', 'F1-Score', 'ROC-AUC'],
    'TRAIN': [
        accuracy_score(y_train, y_pred_xgb_opt_train),
        precision_score(y_train, y_pred_xgb_opt_train),
        recall_score(y_train, y_pred_xgb_opt_train),
        f1_score(y_train, y_pred_xgb_opt_train),
        roc_auc_score(y_train, y_proba_xgb_opt_train)
    ],
    'TEST': [
        accuracy_score(y_test, y_pred_xgb_opt),
        precision_score(y_test, y_pred_xgb_opt),
        recall_score(y_test, y_pred_xgb_opt),
        f1_score(y_test, y_pred_xgb_opt),
        roc_auc_score(y_test, y_proba_xgb_opt)
    ]
})

print(resumen_xgb.to_string(index=False))
print("=" * 70)

In [ ]:
print("\nCOMPARACIÓN: MODELOS OPTIMIZADOS")
print("=" * 80)

modelos_optimizados = {

    'Reg. Logística Optimizada': {
        'y_pred_train': y_pred_lr_opt_train if 'y_pred_lr_opt_train' in globals() else y_pred_lr_train,
        'y_proba_train': y_proba_lr_opt_train if 'y_proba_lr_opt_train' in globals() else y_proba_lr_train,
        'y_pred_test': y_pred_lr_opt,
        'y_proba_test': y_proba_lr_opt,
    },

    'XGBoost Optimizado': {
        'y_pred_train': y_pred_xgb_opt_train if 'y_pred_xgb_opt_train' in globals() else y_pred_xgb_train,
        'y_proba_train': y_proba_xgb_opt_train if 'y_proba_xgb_opt_train' in globals() else y_proba_xgb_train,
        'y_pred_test': y_pred_xgb_opt,
        'y_proba_test': y_proba_xgb_opt,
    },
}

comparativa_final = construir_comparativa_modelos(
    y_train=y_train,
    y_test=y_test,
    modelos=modelos_optimizados,
    orden_modelos=['Reg. Logística', 'Reg. Logística Optimizada', 'XGBoost', 'XGBoost Optimizado']
)

for nombre_modelo in ['Reg. Logística', 'Reg. Logística Optimizada', 'XGBoost', 'XGBoost Optimizado']:
    if nombre_modelo in comparativa_final['Modelo'].values:
        bloque = comparativa_final[comparativa_final['Modelo'] == nombre_modelo]
        print(f"\n{nombre_modelo}")
        print("-" * 80)
        print(bloque[['Métrica', 'Valor TRAIN', 'Valor TEST']].to_string(index=False))

print("=" * 80)